# artifacts: геометрия + CatBoost

Проверяем готовые признаки и локальные выбросы рёбер на прежнем разбиении. Internal test не используем.

Сохранённые выводы относятся к предыдущим запускам. После рефакторинга полное обучение не повторялось.

Порядок запуска: сверху вниз, в отдельном kernel. Данные и локальные модели нужны отдельно от Git. Установка и список файлов — в [инструкции](../../docs/RUNNING.md).

## 1. Данные

In [ ]:
%pip install -q catboost iterative-stratification

In [ ]:
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor
import json
import sys
import numpy as np
import pandas as pd
from catboost import CatBoostClassifier
from iterstrat.ml_stratifiers import MultilabelStratifiedShuffleSplit
from sklearn.metrics import average_precision_score, confusion_matrix, f1_score, precision_score, recall_score
from tqdm.auto import tqdm

try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    pass

ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'data/train.csv').exists()), None)
if ROOT is None:
    ROOT = Path('/content/drive/MyDrive/Colab Notebooks/colab_upload/AI challenge main')
import sys
sys.path.insert(0, str(ROOT))
from src.evaluation import binary_metrics, best_threshold
from src.data import split_indices, load_embeddings
from src.plots import setup_plot_style
setup_plot_style()
assert (ROOT / 'data/train.csv').exists(), ROOT
sys.path.insert(0, str(ROOT / 'labels/artifacts'))
from labels.artifacts.geometry_features import extract_one, FEATURES
from labels.artifacts.component_features import extract_one as extract_components, FEATURES as COMPONENT_FEATURES

labels = pd.read_csv(ROOT / 'data/train.csv')
targets = labels.columns.drop('item_id').tolist()
train_idx, val_idx, test_idx = split_indices(labels)
train = labels.iloc[train_idx].reset_index(drop=True)
val = labels.iloc[val_idx].reset_index(drop=True)
objects = pd.concat([train, val], ignore_index=True)
assert (len(train), len(val), len(test_idx)) == (6259, 1361, 1344)
print('Train:', len(train), 'Validation:', len(val))

## 2. Новые признаки локальных рёбер

Для больших мешей берём до 12 000 граней из четырёх участков. После прерывания расчёт продолжится с сохранённого места.

In [ ]:
out = ROOT / 'labels/artifacts/artifacts'
out.mkdir(parents=True, exist_ok=True)
path = out / 'local_edge_features.csv'
rows = pd.read_csv(path).to_dict('records') if path.exists() else []
done = {row['item_id'] for row in rows}
missing = [item_id for item_id in objects.item_id if item_id not in done]

with ThreadPoolExecutor(max_workers=2) as pool:
    for i, row in enumerate(tqdm(pool.map(lambda item_id: extract_one(item_id, ROOT / 'data/train'), missing),
                                 total=len(missing), desc='Меши'), 1):
        rows.append(row)
        if i % 500 == 0:
            pd.DataFrame(rows).to_csv(path, index=False)
pd.DataFrame(rows).to_csv(path, index=False)
local = pd.DataFrame(rows).set_index('item_id').loc[objects.item_id]
print(local['status'].value_counts())
display(local[FEATURES].describe().round(3))

## 3. Связные компоненты

Считаем по общим индексам вершин и по всем граням меша. Если у соседних граней дублируются вершины, число компонент может быть завышено. Объекты с более чем 500 000 граней или 750 000 вершин пропускаем: частичная выборка дала бы неверное число частей.

In [ ]:
component_path = out / 'component_features.csv'
rows = pd.read_csv(component_path).to_dict('records') if component_path.exists() else []
done = {row['item_id'] for row in rows}
missing = [item_id for item_id in objects.item_id if item_id not in done]

with ThreadPoolExecutor(max_workers=2) as pool:
    for i, row in enumerate(tqdm(pool.map(lambda item_id: extract_components(item_id, ROOT / 'data/train'), missing),
                                 total=len(missing), desc='Компоненты'), 1):
        rows.append(row)
        if i % 250 == 0:
            pd.DataFrame(rows).to_csv(component_path, index=False)
pd.DataFrame(rows).to_csv(component_path, index=False)
components = pd.DataFrame(rows).set_index('item_id').loc[objects.item_id]
print(components['status'].value_counts())
display(components[COMPONENT_FEATURES].describe().round(3))

## 4. Собираем наборы признаков

In [ ]:
config = json.loads((ROOT / 'archive/artifacts/catboost_features_config.json').read_text(encoding='utf-8'))
base_columns = config['feature_columns']
base = pd.read_csv(ROOT / 'archive/artifacts/train_mesh_features_relative.csv').set_index('item_id')
lowpoly = pd.read_csv(ROOT / 'labels/lowpoly/artifacts/geometry_features.csv').set_index('item_id')
noisy = pd.read_csv(ROOT / 'labels/noisy/artifacts/smoothing_features.csv').set_index('item_id')
lowpoly_columns = ['triangle_aspect_p90', 'triangle_near_degenerate_share',
                   'dihedral_over_30_share', 'pca_curvature_p95',
                   'normal_variation_p95', 'boundary_edge_share',
                   'nonmanifold_edge_share', 'duplicate_vertex_share',
                   'invalid_faces', 'degenerate_faces']
noisy_columns = [name for name in noisy if name.startswith('smooth_')]
assert len(base_columns) == 44 and len(noisy_columns) == 14
assert base.index.is_unique and lowpoly.index.is_unique and noisy.index.is_unique
ready = base[base_columns].join(lowpoly[lowpoly_columns]).join(noisy[noisy_columns])
sets = {
    'old_44': base.loc[objects.item_id, base_columns],
    'reused': ready.loc[objects.item_id],
    'reused_local': ready.join(local[FEATURES]).loc[objects.item_id],
    'reused_local_components': ready.join(local[FEATURES]).join(components[COMPONENT_FEATURES]).loc[objects.item_id],
}
for name, X in sets.items():
    assert not np.isinf(X.to_numpy(dtype=float)).any()
    print(name, X.shape)

## 5. CatBoost и метрики

In [ ]:
y_train = train.artifacts.to_numpy()
y_val = val.artifacts.to_numpy()
grid = np.arange(.05, .96, .01)

def evaluate(probability, threshold):
    result = binary_metrics(y_val, probability, threshold)
    return {key: result[key] for key in ['F1', 'Precision', 'Recall', 'AP', 'FP', 'FN', 'Threshold']}

old_model = CatBoostClassifier()
old_model.load_model(str(ROOT / 'archive/models/catboost_features/artifacts.cbm'))
old_p = old_model.predict_proba(sets['old_44'].iloc[len(train):])[:, 1]
old_threshold = config['thresholds'][targets.index('artifacts')]
results = [{'Model': 'old_44', **evaluate(old_p, old_threshold)}]
probabilities = val[['item_id', 'artifacts']].copy()
probabilities['old_44'] = old_p

model_dir = ROOT / 'labels/artifacts/models'
model_dir.mkdir(parents=True, exist_ok=True)
for name in ['reused', 'reused_local', 'reused_local_components']:
    X = sets[name]
    model = CatBoostClassifier(**config['parameters'])
    model.fit(X.iloc[:len(train)], y_train,
              eval_set=(X.iloc[len(train):], y_val),
              early_stopping_rounds=50, verbose=False)
    p = model.predict_proba(X.iloc[len(train):])[:, 1]
    threshold = best_threshold(y_val, p, grid)
    results.append({'Model': name, **evaluate(p, threshold)})
    probabilities[name] = p
    model.save_model(str(model_dir / f'{name}.cbm'))
    print(name, '| F1:', round(results[-1]['F1'], 4))

table = pd.DataFrame(results)
display(table.round(4))
table.to_csv(out / 'geometry_metrics.csv', index=False)
probabilities.to_csv(out / 'geometry_validation.csv', index=False)

## 6. Важные признаки

In [ ]:
best = CatBoostClassifier()
best.load_model(str(model_dir / 'reused_local_components.cbm'))
importance = pd.Series(best.get_feature_importance(), index=sets['reused_local_components'].columns)
display(importance.sort_values(ascending=False).head(15).round(2).to_frame('importance'))
display(importance[COMPONENT_FEATURES].round(2).to_frame('component_importance'))

## Вывод

Старый CatBoost на 44 признаках: F1 = 0.1915. Готовые признаки `lowpoly` и `noisy`: 0.2043. С локальными рёбрами: 0.2133. С добавлением связных компонент: 0.2236. Компоненты уменьшили FP со 136 до 78, но увеличили FN с 41 до 47, а AP снизился с 0.1732 до 0.1509. Поэтому пока нельзя считать этот вариант лучшим. Из 7620 мешей компоненты удалось посчитать для 6442; остальные превысили лимит. Старая связка ResNet18 + CatBoost всё ещё сильнее: F1 = 0.3153. Пороги новых моделей выбирались на одной validation, поэтому F1 может быть оптимистичным. Internal test не использовали.

## Как читать результат

F1 сравниваем на том же split. Порог и вариант модели выбираем по validation; internal test для подбора не используем. Повторный выбор по одной validation может завысить оценку. История и ограничения сохранены в [журнале](../../docs/experiments/artifacts.md). Формулы признаков — в [методике](../../docs/METHODS.md).

## Проверка сохранённой геометрической модели

Эти графики построены после рефакторинга по сохранённым вероятностям и весам. Обучение заново не запускалось. SHAP рассчитан на 128 объектах validation; это важность для модели, а не доказательство причины дефекта.

![Сравнение](../../docs/figures/artifacts_comparison.png)

![SHAP](../../docs/figures/artifacts_shap.png)

![Ошибки геометрической модели](../../docs/figures/artifacts_errors.png)

Числа и источники — в [отчёте](../../docs/RESULTS.md).